# BTCUSD Analyzer – Demo Notebook

This notebook shows how to:
- Load data
- Compute features
- (Optionally) train an ML model
- Run a backtest
- Inspect signals and metrics

> This is for education and research, not financial advice.

In [ ]:
import pandas as pd
import numpy as np
import sys, os

repo_root = os.path.abspath('..')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import config
from data_loader import DataLoader
from features import compute_features
from model import BTCUSDModel
from strategy import generate_signals
from risk_manager import RiskManager
from backtester import Backtester

In [ ]:
# Load or generate data
data_path = config.DATA_FILE
loader = DataLoader(data_path)

# Generate sample data (code test only):
# df = loader.generate_sample_data(config.TRAIN_START, config.TEST_END, config.TIMEFRAME)

# Or load real data:
df = loader.load_from_csv()
print(df.head())

In [ ]:
# Compute features
df = compute_features(df, config)
print(df.columns.tolist())
print(df.tail())

In [ ]:
# Optional: train ML model
if config.ML_ENABLED:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    train_mask = (df['timestamp'] >= config.TRAIN_START) & (df['timestamp'] <= config.TRAIN_END)
    train_df = df.loc[train_mask].copy()
    
    model = BTCUSDModel(config)
    model.train(train_df)
    model.save(config.ML_MODEL_PATH)
    
    ml_probs = model.predict_proba(df)
else:
    ml_probs = None

In [ ]:
# Run backtest
rm = RiskManager(config)
bt = Backtester(df, config, generate_signals, rm)
metrics = bt.run(ml_probs=ml_probs)

for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
# Inspect recent signals
df_sig = generate_signals(df, config, ml_probs=ml_probs)
print(df_sig[['timestamp', 'close', 'signal', 'signal_confidence', 'signal_reason']].tail(10))